# 01 — Ingest & Clean (Superstore)
This notebook:
- Loads raw CSV from `data/raw/`
- Standardises columns and types
- Creates clean dataset for SQL + Power BI
- Outputs to `data/cleaned/`

In [1]:
import os
import pandas as pd
import numpy as np

RAW_PATH = "data/raw/superstore.csv"
OUT_PARQUET = "data/cleaned/superstore_clean.parquet"
OUT_CSV = "data/cleaned/superstore_clean.csv"

os.makedirs("data/cleaned", exist_ok=True)

In [2]:
df = pd.read_csv(RAW_PATH)
df.head(), df.shape

(          Category         City        Country Customer.ID     Customer.Name  \
 0  Office Supplies  Los Angeles  United States   LS-172304  Lycoris Saunders   
 1  Office Supplies  Los Angeles  United States   MV-174854     Mark Van Huff   
 2  Office Supplies  Los Angeles  United States   CS-121304      Chad Sievert   
 3  Office Supplies  Los Angeles  United States   CS-121304      Chad Sievert   
 4  Office Supplies  Los Angeles  United States   AP-109154    Arthur Prichep   
 
    Discount Market  记录数               Order.Date        Order.ID  ... Sales  \
 0       0.0     US    1  2011-01-07 00:00:00.000  CA-2011-130813  ...    19   
 1       0.0     US    1  2011-01-21 00:00:00.000  CA-2011-148614  ...    19   
 2       0.0     US    1  2011-08-05 00:00:00.000  CA-2011-118962  ...    21   
 3       0.0     US    1  2011-08-05 00:00:00.000  CA-2011-118962  ...   111   
 4       0.0     US    1  2011-09-29 00:00:00.000  CA-2011-146969  ...     6   
 
     Segment                Sh

In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace("[ .-]", "_", regex=True)
)

In [4]:
df["order_date"] = pd.to_datetime(df["order_date"])

In [5]:
clean = df.copy()

In [6]:
clean["is_loss"] = (clean["profit"] < 0).astype(int)

In [7]:
# common superstore columns: order_date, ship_date, sales, profit, discount, quantity
for c in ["order_date", "ship_date"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

num_cols = [c for c in ["sales","profit","discount","quantity"] if c in df.columns]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df.isna().mean().sort_values(ascending=False).head(15)

category          0.0
city              0.0
country           0.0
customer_id       0.0
customer_name     0.0
discount          0.0
market            0.0
记录数               0.0
order_date        0.0
order_id          0.0
order_priority    0.0
product_id        0.0
product_name      0.0
profit            0.0
quantity          0.0
dtype: float64

In [8]:
clean = df.copy()

# drop rows missing key business fields
required = [c for c in ["order_id","customer_id","product_id","sales","profit","order_date"] if c in clean.columns]
clean = clean.dropna(subset=required)

# remove negative sales (usually data errors)
if "sales" in clean.columns:
    clean = clean[clean["sales"] >= 0]

# cap extreme discounts to [0, 1] if discount is a fraction
if "discount" in clean.columns:
    clean["discount"] = clean["discount"].clip(lower=0, upper=1)

# create loss flag
clean["is_loss"] = (clean["profit"] < 0).astype(int)

clean.shape

(51290, 28)

In [9]:
# customer cohort month
clean["order_month"] = clean["order_date"].dt.to_period("M").astype(str)

# first order month per customer
first_month = clean.groupby("customer_id")["order_date"].min().dt.to_period("M").astype(str)
clean = clean.join(first_month.rename("customer_first_month"), on="customer_id")

# months since first purchase (roughly)
clean["months_since_first"] = (
    (clean["order_date"].dt.to_period("M").astype(int) - 
     pd.to_datetime(clean["customer_first_month"]).dt.to_period("M").astype(int))
)
clean[["customer_id","order_date","customer_first_month","months_since_first"]].head()

,customer_id,order_date,customer_first_month,months_since_first
0,LS-172304,2011-01-07,2011-01,0
1,MV-174854,2011-01-21,2011-01,0
2,CS-121304,2011-08-05,2011-08,0
3,CS-121304,2011-08-05,2011-08,0
4,AP-109154,2011-09-29,2011-08,1


In [11]:
clean.to_parquet(OUT_PARQUET, index=False)
clean.to_csv(OUT_CSV, index=False)

print("Saved:", OUT_PARQUET)
print("Saved:", OUT_CSV)

Saved: data/cleaned/superstore_clean.parquet
Saved: data/cleaned/superstore_clean.csv
